# 🌫️ Module 2.5: Noise and Denoising — The Core RL Challenge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_02_Image_Processing_Basics/05_Noise_And_Denoising/noise_and_denoising.ipynb)

---

## 🎯 Learning Objectives
1. Types of image noise (Gaussian, salt & pepper, Poisson, speckle)
2. Noise models — mathematical formulation
3. Denoising methods (spatial, frequency, non-local means)
4. Denoising as THE perfect RL problem

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# CIFAR-10: 60,000 real photographs in 10 classes (auto-downloads ~170MB)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10 loaded: {len(cifar10)} real photos (32x32x3)")
print(f"   Classes: {cifar10.classes}")

# scikit-image real test images
sample_images = {
    'astronaut': data.astronaut(),
    'camera': data.camera(),
    'coins': data.coins(),
    'horse': data.horse(),
}
print(f"✅ scikit-image: {len(sample_images)} real test images loaded")

## Deep Mathematical Derivation: Noise Models and Optimal Denoising

### Step 1: Additive Noise Model
The observed image is:
$$I_{\text{noisy}}(x, y) = I_{\text{clean}}(x, y) + \eta(x, y)$$

where $\eta$ is a random noise field.

### Step 2: Gaussian Noise — Complete Statistical Characterization
$$\eta(x, y) \sim \mathcal{N}(0, \sigma^2), \quad p(\eta) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{\eta^2}{2\sigma^2}\right)$$

**Proof that averaging $N$ noisy copies reduces noise by $\sqrt{N}$:**

Let $\bar{I}(x,y) = \frac{1}{N}\sum_{i=1}^N I_{\text{noisy}}^{(i)}(x,y) = I_{\text{clean}} + \frac{1}{N}\sum_{i=1}^N \eta_i$

$$\text{Var}\left[\frac{1}{N}\sum_{i=1}^N \eta_i\right] = \frac{1}{N^2}\sum_{i=1}^N \text{Var}[\eta_i] = \frac{N\sigma^2}{N^2} = \frac{\sigma^2}{N}$$

Standard deviation of noise: $\sigma_{\text{avg}} = \frac{\sigma}{\sqrt{N}}$ $\blacksquare$

### Step 3: Salt-and-Pepper Noise Model
$$I_{\text{noisy}}(x,y) = \begin{cases} 0 & \text{with probability } p/2 \\ 255 & \text{with probability } p/2 \\ I_{\text{clean}}(x,y) & \text{with probability } 1 - p \end{cases}$$

**Proof that median filter is optimal for S&P noise:**
The median minimizes the $L_1$ loss: $\hat{x} = \arg\min_c \sum_i |x_i - c| = \text{median}(x_1, \ldots, x_n)$

For S&P noise: outlier pixels $(0$ or $255)$ are a minority if $p < 0.5$. The median ignores these outliers entirely since it selects the middle value. $\blacksquare$

### Step 4: Poisson (Shot) Noise — Photon Counting
$$P(k \text{ photons}) = \frac{\lambda^k e^{-\lambda}}{k!}, \quad E[k] = \lambda, \quad \text{Var}[k] = \lambda$$

**Key insight:** $\text{SNR} = \frac{E[k]}{\sqrt{\text{Var}[k]}} = \frac{\lambda}{\sqrt{\lambda}} = \sqrt{\lambda}$

**Proof that brighter regions have higher SNR:**
Since $\lambda$ is proportional to light intensity, brighter pixels ($\lambda \gg 1$) have SNR $\propto \sqrt{\lambda}$. Darker regions ($\lambda \approx 1$) have SNR $\approx 1$ — very noisy! $\blacksquare$

### Step 5: PSNR Derivation (Peak Signal-to-Noise Ratio)
$$\text{MSE} = \frac{1}{HW}\sum_{x=0}^{H-1}\sum_{y=0}^{W-1}[I_{\text{clean}}(x,y) - I_{\text{denoised}}(x,y)]^2$$

$$\text{PSNR} = 10 \log_{10}\left(\frac{\text{MAX}^2}{\text{MSE}}\right) = 20\log_{10}(\text{MAX}) - 10\log_{10}(\text{MSE})$$

For 8-bit images: $\text{MAX} = 255$.

**Derivation of the 20/10 coefficients:**
$$10\log_{10}\left(\frac{255^2}{\text{MSE}}\right) = 10\log_{10}(255^2) - 10\log_{10}(\text{MSE}) = 20\log_{10}(255) - 10\log_{10}(\text{MSE}) \quad \blacksquare$$

### Step 6: SSIM — Structural Similarity (Full Derivation)
$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

This decomposes into three components:

**Luminance:** $l(x,y) = \frac{2\mu_x\mu_y + C_1}{\mu_x^2 + \mu_y^2 + C_1}$

**Proof:** $l = 1$ iff $\mu_x = \mu_y$ (same mean brightness). $l$ is maximized when means are equal. $\blacksquare$

**Contrast:** $c(x,y) = \frac{2\sigma_x\sigma_y + C_2}{\sigma_x^2 + \sigma_y^2 + C_2}$

**Structure:** $s(x,y) = \frac{\sigma_{xy} + C_3}{\sigma_x\sigma_y + C_3}$ where $C_3 = C_2/2$

**Proof that $\text{SSIM} = l \cdot c \cdot s$:**
Multiply: $l \cdot c \cdot s = \frac{(2\mu_x\mu_y+C_1) \cdot 2\sigma_x\sigma_y \cdot (\sigma_{xy}+C_3)}{(\mu_x^2+\mu_y^2+C_1)(\sigma_x^2+\sigma_y^2+C_2)(\sigma_x\sigma_y+C_3)}$

With $C_3 = C_2/2$: $2\sigma_x\sigma_y \cdot \frac{\sigma_{xy}+C_2/2}{\sigma_x\sigma_y+C_2/2} = \frac{2\sigma_{xy}+C_2}{1} \cdot \frac{\sigma_x\sigma_y}{\sigma_x\sigma_y+C_2/2}$ ... simplifying yields the SSIM formula. $\blacksquare$

### RL Connection: Denoising as the Perfect RL Problem
- **State:** current (noisy) image $I_t$
- **Action:** denoising operation (filter type, kernel size, strength)
- **Reward:** $R_t = \text{PSNR}(I_{t+1}, I_{\text{target}}) - \text{PSNR}(I_t, I_{\text{target}})$
- **Goal:** maximize cumulative PSNR improvement over sequential denoising steps

This is arguably the most natural image processing task for RL — the agent learns an adaptive, image-specific denoising strategy.

## Maximum Likelihood Estimation of Gaussian Noise Parameters

When presented with a noisy image, the RL agent needs to estimate the noise level to choose the right denoising strategy. MLE provides the optimal parameter estimates.

### Step 1: Likelihood Function for Gaussian Noise

Given the additive model $I_{\text{noisy}}(x,y) = I_{\text{clean}}(x,y) + \eta(x,y)$ where $\eta \sim \mathcal{N}(\mu_\eta, \sigma_\eta^2)$ i.i.d.:

If we observe $N$ noise samples $\eta_1, \ldots, \eta_N$ (e.g., from a flat region of the image where $I_{\text{clean}}$ is approximately constant):

$$\mathcal{L}(\mu_\eta, \sigma_\eta^2) = \prod_{i=1}^N \frac{1}{\sqrt{2\pi}\sigma_\eta} \exp\left(-\frac{(\eta_i - \mu_\eta)^2}{2\sigma_\eta^2}\right)$$

### Step 2: Log-Likelihood

$$\ell(\mu_\eta, \sigma_\eta^2) = -\frac{N}{2}\ln(2\pi) - \frac{N}{2}\ln\sigma_\eta^2 - \frac{1}{2\sigma_\eta^2}\sum_{i=1}^N (\eta_i - \mu_\eta)^2$$

### Step 3: MLE for Mean (Noise Bias)

$$\frac{\partial \ell}{\partial \mu_\eta} = \frac{1}{\sigma_\eta^2}\sum_{i=1}^N (\eta_i - \mu_\eta) = 0$$

$$\hat{\mu}_\eta = \frac{1}{N}\sum_{i=1}^N \eta_i \quad \blacksquare$$

For zero-mean noise (typical case), $\hat{\mu}_\eta \approx 0$.

### Step 4: MLE for Variance (Noise Power)

$$\frac{\partial \ell}{\partial \sigma_\eta^2} = -\frac{N}{2\sigma_\eta^2} + \frac{1}{2\sigma_\eta^4}\sum_{i=1}^N (\eta_i - \mu_\eta)^2 = 0$$

$$\hat{\sigma}_\eta^2 = \frac{1}{N}\sum_{i=1}^N (\eta_i - \hat{\mu}_\eta)^2 \quad \blacksquare$$

### Step 5: Fisher Information and Cramér-Rao Bound

The Fisher information for $\sigma^2$ tells us the best achievable estimation accuracy:

$$I(\sigma^2) = -E\left[\frac{\partial^2 \ell}{\partial (\sigma^2)^2}\right] = \frac{N}{2\sigma^4}$$

The Cramér-Rao lower bound on the variance of any unbiased estimator of $\sigma^2$:

$$\text{Var}(\hat{\sigma}^2) \geq \frac{1}{I(\sigma^2)} = \frac{2\sigma^4}{N}$$

**Practical implication:** To estimate noise level to within 1% accuracy, we need approximately $N = 20{,}000/\sigma^2$ samples from flat image regions.

### Step 6: Robust Noise Estimation via Median Absolute Deviation

In practice, finding pure noise samples is difficult. A robust estimator uses the wavelet domain:

$$\hat{\sigma} = \frac{\text{MAD}(W_{\text{HH}}^{(1)})}{0.6745}$$

where $W_{\text{HH}}^{(1)}$ are the finest-scale diagonal wavelet coefficients and $0.6745 = \Phi^{-1}(3/4)$ is the MAD-to-$\sigma$ conversion factor for Gaussian distributions.

**Proof:** For $X \sim \mathcal{N}(0, \sigma^2)$, $\text{MAD} = \sigma \cdot \Phi^{-1}(3/4) \approx 0.6745\sigma$. $\blacksquare$

## Poisson Noise — Derivation of SNR from Photon Statistics

Poisson noise is the dominant noise source in low-light imaging. Unlike Gaussian noise, its variance is signal-dependent — brighter regions are less noisy (relatively).

### Step 1: Physical Origin

A camera pixel counts photons arriving during the exposure time $T$. If the expected arrival rate is $\lambda$ photons per exposure:

$$P(k \text{ photons}) = \frac{\lambda^k e^{-\lambda}}{k!}$$

### Step 2: Mean and Variance of the Poisson Distribution

**Mean:** $E[k] = \sum_{k=0}^\infty k \cdot \frac{\lambda^k e^{-\lambda}}{k!} = \lambda e^{-\lambda} \sum_{k=1}^\infty \frac{\lambda^{k-1}}{(k-1)!} = \lambda e^{-\lambda} \cdot e^\lambda = \lambda$

**Variance derivation:**
$$E[k^2] = E[k(k-1)] + E[k] = \lambda^2 + \lambda$$

$$E[k(k-1)] = \sum_{k=2}^\infty k(k-1)\frac{\lambda^k e^{-\lambda}}{k!} = \lambda^2 e^{-\lambda}\sum_{k=2}^\infty \frac{\lambda^{k-2}}{(k-2)!} = \lambda^2$$

$$\therefore \text{Var}[k] = E[k^2] - (E[k])^2 = (\lambda^2 + \lambda) - \lambda^2 = \lambda \quad \blacksquare$$

### Step 3: Signal-Dependent SNR

$$\text{SNR} = \frac{E[\text{signal}]}{\text{std}[\text{noise}]} = \frac{\lambda}{\sqrt{\lambda}} = \sqrt{\lambda}$$

**Critical implications for imaging:**

| Expected photons $\lambda$ | SNR | Relative noise $1/\text{SNR}$ |
|:---------------------------|:----|:------------------------------|
| 1 | 1.0 | 100% |
| 10 | 3.2 | 32% |
| 100 | 10.0 | 10% |
| 1000 | 31.6 | 3.2% |
| 10000 | 100.0 | 1% |

**Proof that doubling exposure improves SNR by $\sqrt{2}$:**

If $\lambda_1 = \lambda$ and $\lambda_2 = 2\lambda$, then $\text{SNR}_2 / \text{SNR}_1 = \sqrt{2\lambda}/\sqrt{\lambda} = \sqrt{2}$. $\blacksquare$

### Step 4: Anscombe Transform — Variance Stabilization

The Anscombe transform converts Poisson-distributed data to approximately Gaussian with constant variance:

$$f(x) = 2\sqrt{x + 3/8}$$

**Derivation via the delta method:** If $X \sim \text{Poisson}(\lambda)$, then $g(X) \approx g(\lambda) + g'(\lambda)(X - \lambda)$ for large $\lambda$.

$$\text{Var}[g(X)] \approx [g'(\lambda)]^2 \text{Var}[X] = [g'(\lambda)]^2 \lambda$$

For constant variance: $[g'(\lambda)]^2 \lambda = c \implies g'(\lambda) = c' / \sqrt{\lambda} \implies g(\lambda) = 2c'\sqrt{\lambda}$

Adding $3/8$ inside the square root improves the approximation for small $\lambda$:
$$\text{Var}[2\sqrt{X + 3/8}] \approx 1 \quad \text{for } \lambda \geq 4 \quad \blacksquare$$

### Step 5: Practical Consequence for RL Denoising Agents

A denoising RL agent must adapt its strategy based on signal level:
- **Dark regions** ($\lambda$ small): aggressive denoising needed (low SNR)
- **Bright regions** ($\lambda$ large): light denoising sufficient (high SNR)

The optimal denoising strength is spatially varying: $\sigma_{\text{denoise}}(x,y) \propto 1/\sqrt{I(x,y)}$. An RL agent learns this adaptive behavior automatically through experience.

## 1. Noise Models

### Additive Gaussian Noise:
$$I_{noisy}(x,y) = I_{clean}(x,y) + n(x,y)$$
where $n(x,y) \sim \mathcal{N}(0, \sigma^2)$

### Salt and Pepper Noise:
$$I_{noisy}(x,y) = \begin{cases} 0 & \text{with prob } p/2 \\ 255 & \text{with prob } p/2 \\ I_{clean}(x,y) & \text{with prob } 1-p \end{cases}$$

### Poisson (Shot) Noise:
$$I_{noisy}(x,y) \sim \text{Poisson}(I_{clean}(x,y))$$

### Speckle (Multiplicative) Noise:
$$I_{noisy}(x,y) = I_{clean}(x,y) \cdot (1 + n(x,y))$$
where $n(x,y) \sim \mathcal{N}(0, \sigma^2)$

## 2. Quality Metrics (RL Reward Functions!)

### PSNR (Peak Signal-to-Noise Ratio):
$$\text{PSNR} = 10 \cdot \log_{10}\left(\frac{MAX_I^2}{MSE}\right) = 20 \cdot \log_{10}\left(\frac{MAX_I}{\sqrt{MSE}}\right)$$

where $MSE = \frac{1}{MN}\sum_{i,j}(I_{clean}[i,j] - I_{denoised}[i,j])^2$

### SSIM (Structural Similarity):
$$\text{SSIM}(x,y) = \frac{(2\mu_x\mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$$

## Bilateral Filter — Edge-Preserving Denoising Theory

The bilateral filter is a nonlinear, edge-preserving smoothing filter that bridges the gap between simple Gaussian filtering and the more complex NLM approach.

### Step 1: Definition

$$I_{\text{BF}}(\mathbf{x}) = \frac{1}{W(\mathbf{x})} \sum_{\mathbf{y} \in \mathcal{N}(\mathbf{x})} G_{\sigma_s}(\|\mathbf{x} - \mathbf{y}\|) \cdot G_{\sigma_r}(|I(\mathbf{x}) - I(\mathbf{y})|) \cdot I(\mathbf{y})$$

where $W(\mathbf{x}) = \sum_{\mathbf{y}} G_{\sigma_s}(\|\mathbf{x}-\mathbf{y}\|) \cdot G_{\sigma_r}(|I(\mathbf{x})-I(\mathbf{y})|)$ is the normalization.

**Two Gaussian kernels:**
- **Spatial kernel** $G_{\sigma_s}$: nearby pixels contribute more (like regular Gaussian blur)
- **Range kernel** $G_{\sigma_r}$: similar-intensity pixels contribute more (preserves edges)

### Step 2: Edge-Preserving Property — Proof

**Theorem:** At an edge where $|I(\mathbf{x}) - I(\mathbf{y})| \gg \sigma_r$, pixels across the edge are effectively excluded from the average.

**Proof:** When $|I(\mathbf{x}) - I(\mathbf{y})| = 3\sigma_r$:
$$G_{\sigma_r}(3\sigma_r) = \exp\left(-\frac{9\sigma_r^2}{2\sigma_r^2}\right) = e^{-4.5} \approx 0.011$$

The pixel across the edge gets weight $\approx 1\%$ — effectively zero. Only same-side pixels contribute to the average, preventing cross-edge blurring. $\blacksquare$

### Step 3: Relationship to Anisotropic Diffusion

The bilateral filter can be interpreted as a single step of Perona-Malik anisotropic diffusion:

$$\frac{\partial I}{\partial t} = \text{div}(g(|\nabla I|) \nabla I)$$

where $g(s) = \exp(-s^2 / 2\sigma_r^2)$ is the edge-stopping function.

**Proof sketch:** Discretizing the divergence operator and approximating the gradient gives a weighted sum of neighbors, with weights that depend on intensity differences — matching the bilateral filter structure. $\blacksquare$

### Step 4: Parameter Selection Theory

**$\sigma_s$ (spatial):** Controls the size of the averaging region
- Small $\sigma_s$: less smoothing, preserves fine detail
- Large $\sigma_s$: more smoothing, removes more noise

**$\sigma_r$ (range):** Controls the edge sensitivity
- Small $\sigma_r$: only very similar pixels are averaged → many edges preserved but weak denoising
- Large $\sigma_r$: wider range of intensities averaged → stronger denoising but edges may blur

**Optimal setting:** $\sigma_r \approx 1.5\sigma_{\text{noise}}$ balances denoising and edge preservation.

### Step 5: Computational Complexity

Naive bilateral: $O(N \cdot |\mathcal{N}|)$ per pixel — expensive for large kernels.

**Fast bilateral filter** (Paris & Durand, 2006): Lift the image to a 3D space $(x, y, I)$, apply linear filtering in this space, then "slice" back:
$$\text{Complexity: } O(N \cdot \sigma_s/\sigma_r)$$

This makes the bilateral filter practical for real-time RL applications, where the agent must process frames at video rate.

In [ ]:
# Create a clean sample image
size = 128
x = np.linspace(-3, 3, size)
X, Y = np.meshgrid(x, x)
clean = np.zeros((size, size), dtype=np.float64)

# Multiple features
clean += 150 * np.exp(-(X**2 + Y**2) / 1.5)  # Center blob
clean += 80 * (np.sin(X*3) > 0).astype(float)  # Stripes
clean += 50 * ((X-1.5)**2 + (Y-1.5)**2 < 0.5).astype(float)  # Small circle
clean = np.clip(clean, 0, 255)

# Add different types of noise
np.random.seed(42)

# Gaussian noise
sigma = 25
gaussian_noise = clean + np.random.normal(0, sigma, clean.shape)
gaussian_noise = np.clip(gaussian_noise, 0, 255)

# Salt and pepper
sp_noise = clean.copy()
prob = 0.05
salt = np.random.random(clean.shape) < prob/2
pepper = np.random.random(clean.shape) < prob/2
sp_noise[salt] = 255
sp_noise[pepper] = 0

# Speckle noise
speckle_noise = clean * (1 + np.random.normal(0, 0.2, clean.shape))
speckle_noise = np.clip(speckle_noise, 0, 255)

# Poisson noise
poisson_noise = np.random.poisson(clean.clip(1, None)).astype(np.float64)
poisson_noise = np.clip(poisson_noise, 0, 255)

# Visualize noise types
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

noisy_images = [
    (clean, 'Clean Original'),
    (gaussian_noise, f'Gaussian (σ={sigma})'),
    (sp_noise, f'Salt & Pepper (p={prob})'),
    (speckle_noise, 'Speckle (σ=0.2)'),
    (poisson_noise, 'Poisson (Shot)'),
]

for idx, (img, title) in enumerate(noisy_images):
    row, col = idx // 3, idx % 3
    axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    if idx > 0:
        p = psnr(clean, img, data_range=255)
        axes[row, col].set_title(f'{title}\nPSNR: {p:.1f} dB')
    else:
        axes[row, col].set_title(title)
    axes[row, col].axis('off')

# Reward explanation
axes[1, 2].text(0.5, 0.5,
    'RL Reward Functions:\n\n'
    '• PSNR ↑ = better denoising\n'
    '• SSIM ↑ = preserved structure\n\n'
    'Agent Goal:\n'
    'Choose denoising method\n'
    'that maximizes BOTH!',
    ha='center', va='center', fontsize=12,
    bbox=dict(boxstyle='round', facecolor='lightcyan'),
    transform=axes[1, 2].transAxes)
axes[1, 2].axis('off')

plt.suptitle('Types of Image Noise — Each Needs Different Denoising Strategy',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('noise_types.png', dpi=150, bbox_inches='tight')
plt.show()

## Wiener Filter — Derivation from the MMSE Principle

The Wiener filter is the optimal linear denoising filter in the minimum mean square error (MMSE) sense. Its derivation elegantly combines estimation theory with frequency-domain analysis.

### Step 1: Problem Formulation

**Observation model:** $Y = X + N$ where:
- $X$: clean image (unknown)
- $N$: noise (zero-mean, variance $\sigma_n^2$, independent of $X$)
- $Y$: observed noisy image

**Goal:** Find a linear filter $H$ that minimizes:
$$\text{MSE} = E[\|X - \hat{X}\|^2] = E[\|X - H \cdot Y\|^2]$$

where multiplication is in the frequency domain (pointwise).

### Step 2: Derivation in the Frequency Domain

In the Fourier domain, at each frequency $(u, v)$:
$$\hat{X}(u,v) = H(u,v) \cdot Y(u,v)$$

The MSE at frequency $(u,v)$:
$$\epsilon(u,v) = E[|X(u,v) - H(u,v)Y(u,v)|^2]$$

### Step 3: Optimization

Expanding:
$$\epsilon = E[|X|^2] - H^*E[XY^*] - HE[X^*Y] + |H|^2E[|Y|^2]$$

Since $Y = X + N$ and $X \perp N$:
- $E[|Y|^2] = S_{XX} + S_{NN}$ (power spectral densities)
- $E[XY^*] = E[X(X^* + N^*)] = S_{XX}$

Taking derivative with respect to $H^*$ and setting to zero:
$$\frac{\partial \epsilon}{\partial H^*} = -S_{XX} + H(S_{XX} + S_{NN}) = 0$$

### Step 4: The Wiener Filter

$$\boxed{H_{\text{Wiener}}(u,v) = \frac{S_{XX}(u,v)}{S_{XX}(u,v) + S_{NN}(u,v)} = \frac{|X(u,v)|^2}{|X(u,v)|^2 + \sigma_n^2}}$$

**Proof that this is the MMSE estimator:**

The second derivative $\frac{\partial^2 \epsilon}{\partial H \partial H^*} = S_{XX} + S_{NN} > 0$, confirming this is a minimum. $\blacksquare$

### Step 5: Interpretation

The Wiener filter is a **frequency-dependent gain**:

$$H_{\text{Wiener}}(u,v) = \frac{\text{SNR}(u,v)}{1 + \text{SNR}(u,v)}$$

where $\text{SNR}(u,v) = S_{XX}(u,v)/S_{NN}(u,v)$ is the spectral signal-to-noise ratio.

**Behavior:**
- Where SNR is high (strong signal): $H \approx 1$ (pass through — signal dominates)
- Where SNR is low (mostly noise): $H \approx 0$ (suppress — noise dominates)
- Transition: smooth interpolation between keeping and suppressing

### Step 6: Connection to Regularization

The Wiener filter can be rewritten as:
$$\hat{X} = \arg\min_X \|Y - X\|^2 + \lambda \|\mathbf{L} X\|^2$$

where $\lambda = \sigma_n^2$ and $\mathbf{L}$ is chosen so $|\hat{L}|^2 = S_{XX}^{-1}$. This reveals that Wiener filtering is equivalent to **Tikhonov regularization** — a constrained optimization balancing data fidelity against a smoothness prior.

### Practical Limitation

The Wiener filter requires knowledge of $S_{XX}$ (the clean image power spectrum), which is unknown. In practice, it is estimated from the noisy image or a generic model for natural images: $S_{XX}(f) \propto f^{-2}$ (the famous $1/f^2$ power law of natural images).

In [ ]:
# Denoising methods comparison
noisy = gaussian_noise.astype(np.uint8)

# Method 1: Gaussian blur
denoised_gauss = cv2.GaussianBlur(noisy, (5, 5), 1.5)

# Method 2: Median filter
denoised_median = cv2.medianBlur(noisy, 5)

# Method 3: Bilateral filter (edge-preserving)
denoised_bilateral = cv2.bilateralFilter(noisy, 9, 75, 75)

# Method 4: Non-local means
denoised_nlm = cv2.fastNlMeansDenoising(noisy, None, 30, 7, 21)

# Compare
methods = [
    (noisy, 'Noisy Input'),
    (denoised_gauss, 'Gaussian Blur'),
    (denoised_median, 'Median Filter'),
    (denoised_bilateral, 'Bilateral Filter'),
    (denoised_nlm, 'Non-Local Means'),
    (clean.astype(np.uint8), 'Ground Truth'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, (img, title) in enumerate(methods):
    row, col = idx // 3, idx % 3
    axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=255)

    p = psnr(clean.astype(np.uint8), img.astype(np.uint8), data_range=255)
    s = ssim(clean.astype(np.uint8), img.astype(np.uint8), data_range=255)
    axes[row, col].set_title(f'{title}\nPSNR={p:.1f}dB, SSIM={s:.3f}')
    axes[row, col].axis('off')

plt.suptitle('Denoising Methods — Which is Best? Let RL Decide!\n'
             '(No single method wins for all images)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('denoising_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🤖 KEY INSIGHT: No single denoising method is best for all images!")
print("   → RL agent learns WHICH method to apply for EACH image!")
print("   → This is why RL is powerful for image processing!")

## Non-Local Means (NLM) — Weight Function Derivation from Patch Distances

Non-Local Means was a breakthrough in denoising because it exploits the self-similarity of natural images rather than relying on local smoothness.

### Step 1: Key Insight — Self-Similarity

Natural images contain many similar patches in different locations. NLM averages similar patches to denoise, even when they are far apart spatially.

### Step 2: Patch Comparison

Define a square patch of size $(2d+1) \times (2d+1)$ centered at pixel $\mathbf{i}$:
$$\mathcal{P}_\mathbf{i} = \{I(\mathbf{i} + \mathbf{k}) : \mathbf{k} \in [-d, d]^2\}$$

The squared patch distance between pixels $\mathbf{i}$ and $\mathbf{j}$:
$$d^2(\mathbf{i}, \mathbf{j}) = \frac{1}{(2d+1)^2} \sum_{\mathbf{k} \in [-d,d]^2} (I(\mathbf{i} + \mathbf{k}) - I(\mathbf{j} + \mathbf{k}))^2$$

### Step 3: Weight Function Derivation

**Requirement:** Similar patches should contribute strongly; dissimilar patches should be suppressed.

**Natural choice:** Gaussian weighting on patch distance:
$$w(\mathbf{i}, \mathbf{j}) = \frac{1}{Z(\mathbf{i})} \exp\left(-\frac{d^2(\mathbf{i}, \mathbf{j})}{h^2}\right)$$

where $h$ is the filtering parameter controlling the decay rate, and $Z(\mathbf{i}) = \sum_\mathbf{j} \exp(-d^2(\mathbf{i}, \mathbf{j})/h^2)$ is the normalization constant.

**Derivation from Bayesian framework:**

Assume that similar patches come from the same underlying clean signal plus i.i.d. noise:
$$\mathcal{P}_\mathbf{j}^{\text{noisy}} = \mathcal{P}_\mathbf{i}^{\text{clean}} + \boldsymbol{\eta}_\mathbf{j}$$

The likelihood that patch $\mathbf{j}$ has the same clean content as $\mathbf{i}$:
$$P(\mathcal{P}_\mathbf{j} | \mathcal{P}_\mathbf{j} \text{ matches } \mathcal{P}_\mathbf{i}) \propto \exp\left(-\frac{\|\mathcal{P}_\mathbf{i} - \mathcal{P}_\mathbf{j}\|^2}{2\sigma^2 \cdot (2d+1)^2}\right)$$

Setting $h^2 = 2\sigma^2$ recovers the NLM weights. $\blacksquare$

### Step 4: NLM Estimate

$$\hat{I}(\mathbf{i}) = \sum_{\mathbf{j} \in \mathcal{S}} w(\mathbf{i}, \mathbf{j}) \cdot I(\mathbf{j})$$

where $\mathcal{S}$ is the search window (neighborhood of $\mathbf{i}$ where similar patches are sought).

### Step 5: Proof That NLM Is the MMSE Estimator Under Self-Similarity

**Theorem:** Under the assumption that the image contains multiple copies of each patch corrupted by i.i.d. Gaussian noise, NLM is the minimum mean square error estimator.

**Proof (sketch):** The posterior mean (MMSE estimate) of the clean value $I_{\text{clean}}(\mathbf{i})$ given all observations is:

$$E[I_{\text{clean}}(\mathbf{i}) | I_{\text{noisy}}] = \sum_\mathbf{j} P(\mathbf{j} \text{ matches } \mathbf{i} | I_{\text{noisy}}) \cdot I(\mathbf{j})$$

The posterior matching probability is proportional to the Gaussian weight $w(\mathbf{i}, \mathbf{j})$, yielding exactly the NLM formula. $\blacksquare$

### Step 6: Parameter Selection

**Patch size** $(2d+1)$: larger patches → more robust matching but slower
- Typical: $7 \times 7$ or $5 \times 5$

**Filter parameter** $h$: controls denoising strength
- $h \approx \sigma$ (noise standard deviation) is optimal
- $h \ll \sigma$: insufficient denoising
- $h \gg \sigma$: over-smoothing

**Search window** $|\mathcal{S}|$: larger → better quality but $O(|\mathcal{S}| \cdot (2d+1)^2)$ per pixel
- Typical: $21 \times 21$

An RL agent can learn to set $(d, h, |\mathcal{S}|)$ adaptively based on image content — something traditional NLM cannot do.

## Total Variation Denoising — Euler-Lagrange Derivation

Total Variation (TV) denoising preserves edges while removing noise, by penalizing the total variation (sum of gradient magnitudes) of the image.

### Step 1: The TV Denoising Objective

$$\hat{I} = \arg\min_I \underbrace{\frac{1}{2}\int\int (I(x,y) - I_{\text{noisy}}(x,y))^2 \, dx\,dy}_{\text{data fidelity}} + \lambda \underbrace{\int\int |\nabla I(x,y)| \, dx\,dy}_{\text{total variation}}$$

where $|\nabla I| = \sqrt{I_x^2 + I_y^2}$ is the gradient magnitude.

### Step 2: Why TV Preserves Edges (Unlike $L^2$ Regularization)

**$L^2$ penalty** $\|\nabla I\|_2^2 = \int (I_x^2 + I_y^2) dx\,dy$ penalizes large gradients quadratically → blurs edges.

**$L^1$ (TV) penalty** $\|\nabla I\|_1 = \int |\nabla I| dx\,dy$ penalizes large gradients linearly → permits sharp transitions.

**Proof by analogy with 1D denoising:**

For 1D: $\min_u \frac{1}{2}(u - v)^2 + \lambda |u'|$

The solution is the soft-thresholding of the derivative: edges with $|v'| > \lambda$ are preserved (reduced by $\lambda$), while small variations ($|v'| < \lambda$) are eliminated entirely. $\blacksquare$

### Step 3: Euler-Lagrange Equation Derivation

The functional to minimize:
$$E[I] = \frac{1}{2}\int\int (I - I_0)^2 + \lambda\sqrt{I_x^2 + I_y^2} \, dx\,dy$$

The Euler-Lagrange equation gives the necessary condition for a minimizer:

$$\frac{\partial F}{\partial I} - \frac{\partial}{\partial x}\frac{\partial F}{\partial I_x} - \frac{\partial}{\partial y}\frac{\partial F}{\partial I_y} = 0$$

where $F = \frac{1}{2}(I - I_0)^2 + \lambda\sqrt{I_x^2 + I_y^2}$.

Computing each term:
$$\frac{\partial F}{\partial I} = I - I_0$$

$$\frac{\partial F}{\partial I_x} = \frac{\lambda I_x}{\sqrt{I_x^2 + I_y^2}}, \quad \frac{\partial F}{\partial I_y} = \frac{\lambda I_y}{\sqrt{I_x^2 + I_y^2}}$$

### Step 4: The TV Diffusion Equation

The Euler-Lagrange equation becomes:

$$I - I_0 = \lambda \cdot \text{div}\left(\frac{\nabla I}{|\nabla I|}\right) = \lambda \cdot \kappa \cdot |\nabla I|$$

where $\kappa$ is the curvature of the level sets of $I$, and $\text{div}(\nabla I / |\nabla I|)$ is the mean curvature motion operator.

**Interpretation:** TV denoising smooths along edges (low curvature) while preserving edges themselves (high gradient magnitude). This is mathematically equivalent to evolving the image by **mean curvature flow**, which shrinks level curves at a rate proportional to their curvature. $\blacksquare$

### Step 5: Discrete Implementation (Split-Bregman / ADMM)

The non-differentiability of $|\nabla I|$ at $\nabla I = 0$ makes direct optimization difficult. The split-Bregman method introduces auxiliary variables:

$$\min_{I, \mathbf{d}} \frac{1}{2}\|I - I_0\|^2 + \lambda\|\mathbf{d}\|_1 + \frac{\mu}{2}\|\mathbf{d} - \nabla I - \mathbf{b}\|^2$$

This splits into two subproblems that can be solved efficiently:
1. **I-update:** Solve a linear system (Poisson equation) — $O(N\log N)$ via FFT
2. **d-update:** Pointwise soft-thresholding — $O(N)$

### Step 6: RL Agent with TV as Prior

An RL denoising agent can use TV regularization strength $\lambda$ as a **learned action parameter**:
- High $\lambda$: aggressive denoising, potential over-smoothing of texture
- Low $\lambda$: preserves detail but retains noise
- The agent learns to set $\lambda$ spatially, applying strong TV in flat regions and weak TV in textured regions

## BM3D — Block-Matching and 3D Collaborative Filtering Theory

BM3D (Dabov et al., 2007) is considered the gold standard of traditional denoising methods. Its mathematical framework combines ideas from patch matching, transform-domain shrinkage, and collaborative filtering.

### Step 1: Algorithm Overview

BM3D operates in two stages, each with three substeps:

**Stage 1 (Basic estimate):**
1. **Block matching:** For each reference patch, find similar patches
2. **3D transform:** Stack similar patches into a 3D group, apply 3D transform (2D spatial + 1D across patches)
3. **Shrinkage:** Apply hard thresholding in the 3D transform domain
4. **Inverse transform + aggregation:** Transform back and average overlapping estimates

**Stage 2 (Wiener filtering):** Repeat with the Stage 1 estimate as a pilot, using Wiener coefficients instead of hard thresholding.

### Step 2: Block Matching — Finding Similar Patches

For reference patch $\mathcal{P}_\mathbf{x}$, the set of similar patches is:

$$S(\mathbf{x}) = \{\mathbf{y} : d(\mathcal{P}_\mathbf{x}, \mathcal{P}_\mathbf{y}) \leq \tau\}$$

where $d$ is the squared $L^2$ distance. This is the same concept as NLM, but BM3D uses it differently (grouping rather than weighted averaging).

### Step 3: 3D Collaborative Filtering

Stack the $|S(\mathbf{x})|$ similar patches into a 3D array $\mathbf{G} \in \mathbb{R}^{p \times p \times |S|}$.

Apply a 3D linear transform $\mathcal{T}_{3D}$:
$$\hat{\mathbf{G}} = \mathcal{T}_{3D}(\mathbf{G})$$

Typically: $\mathcal{T}_{3D}$ = 2D DCT (spatial) + 1D Haar/DCT (across patches).

### Step 4: Hard Thresholding in Transform Domain

Apply hard thresholding with threshold $\lambda\sigma$:
$$\hat{G}_{ijk}^{\text{HT}} = \begin{cases} \hat{G}_{ijk} & |\hat{G}_{ijk}| \geq \lambda\sigma \\ 0 & |\hat{G}_{ijk}| < \lambda\sigma \end{cases}$$

**Why this works:** In the 3D transform domain, signal energy concentrates in a few large coefficients, while noise energy spreads uniformly across all coefficients. Thresholding removes the noise-dominated coefficients.

**Proof of near-optimality:** For Gaussian noise and a sparse signal in the transform basis, hard thresholding with $\lambda = \sqrt{2\ln N}$ (universal threshold) achieves:
$$E[\|\hat{x} - x\|^2] \leq (2\ln N + 1) \sum_{i=1}^N \min(\theta_i^2, \sigma^2)$$

where $\theta_i$ are the true coefficients — within a $\log N$ factor of the oracle (ideal) risk. $\blacksquare$

### Step 5: Why 3D Grouping Helps

**Theorem:** Collaborative filtering in 3D achieves better denoising than independent 2D filtering of each patch.

**Proof sketch:** When $M$ similar patches are stacked, the signal is the same in each patch (approximately), while the noise is independent. The transform along the "match" dimension averages the $M$ copies, reducing noise by $\sqrt{M}$:

$$\sigma_{\text{3D}} = \frac{\sigma}{\sqrt{M}} \quad \text{along the averaging direction}$$

This $\sqrt{M}$-fold SNR improvement enables more aggressive thresholding with less signal loss. $\blacksquare$

### Step 6: BM3D vs. Deep Learning

BM3D achieves PSNR within 0.5-1.5 dB of the best deep learning denoisers (DnCNN, FFDNet) — remarkable for a non-learned method. An RL framework can bridge the gap: use BM3D as a strong baseline action and let the agent learn when to apply learned corrections:

$$I_{\text{output}} = \alpha \cdot I_{\text{BM3D}} + (1-\alpha) \cdot I_{\text{DnCNN}}$$

where $\alpha$ is an RL-learned spatially-varying blending factor.

## Summary

### Module 2 Complete!

We now have the full image processing toolkit:
- Filters & Convolutions → RL actions
- Edge Detection → RL reward signals
- Morphological Operations → Shape-based RL actions
- Geometric Transforms → Spatial RL actions
- Denoising → The perfect RL challenge

**Next: Module 3 — RL Mathematical Foundations** (where we formalize everything!)

---